## Merge Review Results

In [1]:
import json

In [2]:
def merge_human_agreements(files):
    merged = files[0]

    id_to_sample = {
        s["box_id"]: s for s in merged["all_samples"]
    }

    for file in files[1:]:
        for sample in file["all_samples"]:
            box_id = sample["box_id"]

            if box_id not in id_to_sample:
                continue 

            target = id_to_sample[box_id]

            src = sample.get("human_class_agreement", {})
            tgt = target.get("human_class_agreement", {})

            for reviewer, value in src.items():
                tgt[reviewer] = value

            target["human_class_agreement"] = tgt

    return merged

In [ ]:
dataset = "Cityscapes"
    
file1 = json.load(open(f"review_decisions/{dataset}_class_or_not_review_reviewer_3.json"))
file2 = json.load(open(f"review_decisions/{dataset}_class_or_not_review_reviewer_2.json"))
file3 = json.load(open(f"review_decisions/{dataset}_class_or_not_review_reviewer_1.json"))

merged = merge_human_agreements([file1, file2])

with open(f"review_decisions/{dataset}_class_or_not_review_merged.json", "w") as f:
    json.dump(merged, f, indent=4)

Write results in unbiased_samples datasets

In [5]:
"""Cityscapes, Kitti, PascalVOC"""
dataset = "PascalVOC"
reviewer = ["reviewer_3", "reviewer_2", "reviewer_1"]

out_path = f"/path/to/benchmark_paper_post_processing/our_datasets/unbiased_samples/{dataset}_sample_with_unbiased_annotations.json"

review_file = json.load(open(f"review_decisions/{dataset}_class_or_not_review_merged.json"))
output_file = json.load(open(out_path))

all_samples = review_file["all_samples"]
all_objects = output_file["objects"]

for sample in all_samples:
    img_name = sample["image_name"]
    bbox = sample["bbox"]
    softlabel = sample["soft_label"]
    drawn_prob = sample["drawn_prob"]
    drawn_class = sample["drawn_class_name"]
    annotation_index = sample["annotation_index"]
    human_class_agreement = sample["human_class_agreement"]
    
    review_results = []
    
    for rev in reviewer:
        review_results.append(human_class_agreement[rev])

    if dataset == "Kitti" or dataset == "Cityscapes":
        img_name = img_name.replace(".png", "")
    
    rel_obj = all_objects[img_name][annotation_index]
    rel_obj["human_calibration"] = {
        "drawn_prob": drawn_prob,
        "drawn_class_name": drawn_class,
        "review_results": review_results
    }
    
output_file["objects"] = all_objects

with open(out_path, "w") as f:
    json.dump(output_file, f, indent=4)



In [6]:
dataset = "COCO"
reviewer = ["reviewer_3", "reviewer_2", "reviewer_1"]
categories = ["Accessory", "Animal", "Electronic", "Food", "Furniture", "Indoor", "KitchenAppliance", "Outdoor", "Person", "Sports", "Vehicle"]

out_path = f"/path/to/benchmark_paper_post_processing/our_datasets/unbiased_samples/{dataset}_CATEGORY_sample_with_unbiased_annotations.json"

review_file = json.load(open(f"review_decisions/{dataset}_class_or_not_review_merged.json"))

for category in categories:
    output_file = json.load(open(out_path.replace("CATEGORY", category)))

    all_samples = review_file["all_samples"]
    all_objects = output_file["objects"]

    for sample in all_samples:
        img_name = sample["image_name"]
        bbox = sample["bbox"]
        softlabel = sample["soft_label"]
        drawn_prob = sample["drawn_prob"]
        drawn_class = sample["drawn_class_name"]
        annotation_index = sample["annotation_index"]
        human_class_agreement = sample["human_class_agreement"]
        
        review_results = []
        
        for rev in reviewer:
            review_results.append(human_class_agreement[rev])
            
        if img_name in all_objects:
            img_objs = all_objects[img_name]
            
            if annotation_index < len(img_objs):
                rel_obj = all_objects[img_name][annotation_index]
                
                if rel_obj["bbox"] == bbox and rel_obj["soft_label"] == softlabel:
                    print(rel_obj)
                    rel_obj["human_calibration"] = {
                        "drawn_prob": drawn_prob,
                        "drawn_class_name": drawn_class,
                        "review_results": review_results
                    }
        
    output_file["objects"] = all_objects

    with open(out_path.replace("CATEGORY", category), "w") as f:
        json.dump(output_file, f, indent=4)

{'mo_id': 'fc994bff-f9b4-4b2e-aeea-1ab1e9432fe9', 'bbox': [121, 296, 235, 372], 'frequencies': {'person': 0, 'bicycle': 0, 'car': 0, 'motorcycle': 0, 'airplane': 0, 'bus': 0, 'train': 0, 'truck': 0, 'boat': 0, 'trafficlight': 0, 'firehydrant': 0, 'stopsign': 0, 'parkingmeter': 0, 'bench': 0, 'bird': 0, 'cat': 0, 'dog': 0, 'horse': 0, 'sheep': 0, 'cow': 0, 'elephant': 0, 'bear': 0, 'zebra': 0, 'giraffe': 0, 'backpack': 17, 'umbrella': 0, 'handbag': 1, 'tie': 0, 'suitcase': 3, 'frisbee': 0, 'skis': 0, 'snowboard': 0, 'sportsball': 0, 'kite': 0, 'baseballbat': 0, 'baseballglove': 0, 'skateboard': 0, 'surfboard': 0, 'tennisracket': 0, 'bottle': 0, 'wineglass': 0, 'cup': 0, 'fork': 0, 'knife': 0, 'spoon': 0, 'bowl': 0, 'banana': 0, 'apple': 0, 'sandwich': 0, 'orange': 0, 'broccoli': 0, 'carrot': 0, 'hotdog': 0, 'pizza': 0, 'donut': 0, 'cake': 0, 'chair': 0, 'couch': 0, 'pottedplant': 0, 'bed': 0, 'diningtable': 0, 'toilet': 0, 'tv': 0, 'laptop': 0, 'mouse': 0, 'remote': 0, 'keyboard': 0, 'c